In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import numpy as np
import pandas as pd
import pickle
import matplotlib.pyplot as plt
import tensorflow as tf

# Define file path
BASE_DIR = '/content/drive/MyDrive/fake_news_data'
data_path = os.path.join(BASE_DIR, 'combined_fake_news_preprocessed.csv')

# Create model directory inside Drive
models_dir = os.path.join(BASE_DIR, 'models')
os.makedirs(models_dir, exist_ok=True)

print("Drive mounted. Dataset will be loaded from:", data_path)

In [ ]:
df = pd.read_csv(data_path)
df = df[['cleaned_text', 'label']].copy()
df = df[df['cleaned_text'].str.strip() != '']   # remove empty
print("Dataset shape:", df.shape)
print("Label distribution:\n", df['label'].value_counts())

In [ ]:
from sklearn.model_selection import train_test_split

X = df['cleaned_text'].values
y = df['label'].values

# Use same random state to keep consistency
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.125, random_state=42, stratify=y_temp
)

print(f"Train: {len(X_train)}, Val: {len(X_val)}, Test: {len(X_test)}")

In [ ]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Hyperparameters
MAX_NB_WORDS = 20000
MAX_SEQUENCE_LENGTH = 500

tokenizer = Tokenizer(num_words=MAX_NB_WORDS, oov_token='<OOV>')
tokenizer.fit_on_texts(X_train)

X_train_seq = tokenizer.texts_to_sequences(X_train)
X_val_seq   = tokenizer.texts_to_sequences(X_val)
X_test_seq  = tokenizer.texts_to_sequences(X_test)

X_train_pad = pad_sequences(X_train_seq, maxlen=MAX_SEQUENCE_LENGTH, padding='post', truncating='post')
X_val_pad   = pad_sequences(X_val_seq,   maxlen=MAX_SEQUENCE_LENGTH, padding='post', truncating='post')
X_test_pad  = pad_sequences(X_test_seq,  maxlen=MAX_SEQUENCE_LENGTH, padding='post', truncating='post')

print("Training shape:", X_train_pad.shape)
print("Vocabulary size:", len(tokenizer.word_index) + 1)

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Bidirectional, LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

EMBEDDING_DIM = 100

model = Sequential([
    Embedding(input_dim=MAX_NB_WORDS, output_dim=EMBEDDING_DIM),
    Bidirectional(LSTM(64, dropout=0.2, recurrent_dropout=0.2, return_sequences=False)),
    Dense(64, activation='relu'),
    Dropout(0.5),
    Dense(1, activation='sigmoid')
])

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model.summary()

In [ ]:
early_stop = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)
checkpoint = ModelCheckpoint(
    os.path.join(models_dir, 'bilstm_best.keras'),
    monitor='val_accuracy',
    save_best_only=True
)

history = model.fit(
    X_train_pad, y_train,
    validation_data=(X_val_pad, y_val),
    epochs=20,
    batch_size=64,
    callbacks=[early_stop, checkpoint],
    verbose=1
)

In [ ]:
# Load best model (in case early stopping restored earlier weights)
model.load_weights(os.path.join(models_dir, 'bilstm_best.keras'))

y_pred_prob = model.predict(X_test_pad)
y_pred = (y_pred_prob > 0.5).astype(int).flatten()

from sklearn.metrics import classification_report, accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

cm = confusion_matrix(y_test, y_pred)
print("Confusion Matrix:")
print(cm)
print("\nBiLSTM Test Set Evaluation:")
print(classification_report(y_test, y_pred, target_names=['Real', 'Fake']))

acc = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred)
rec = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print(f"Accuracy:  {acc:.4f}")
print(f"Precision: {prec:.4f}")
print(f"Recall:    {rec:.4f}")
print(f"F1-score:  {f1:.4f}")

In [ ]:
# Save tokenizer for later use in Streamlit
with open(os.path.join(models_dir, 'tokenizer.pkl'), 'wb') as f:
    pickle.dump(tokenizer, f)

# Save final model
model.save(os.path.join(models_dir, 'bilstm_final.keras'))

print("Model and tokenizer saved to Drive.")

In [ ]:
save_path = os.path.join(models_dir, 'bilstm_savedmodel')

# Load existing .keras model
model = tf.keras.models.load_model(os.path.join(models_dir, 'bilstm_best.keras'))

# Save as SavedModel (directory format)
model.export(save_path)
print("Model saved as SavedModel in", save_path)